In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [21]:
df = pd.read_csv("./job_applicant_dataset.csv")

In [22]:
df.head()

,Job Applicant Name,Age,Gender,Race,Ethnicity,Resume,Job Roles,Job Description,Best Match
0,Daisuke Mori,29,Male,Mongoloid/Asian,Vietnamese,"Proficient in Injury Prevention, Motivation, N...",Fitness Coach,A Fitness Coach is responsible for helping cl...,0
1,Taichi Shimizu,31,Male,Mongoloid/Asian,Filipino,"Proficient in Healthcare, Pharmacology, Medica...",Physician,"Diagnose and treat illnesses, prescribe medica...",0
2,Sarah Martin,46,Female,White/Caucasian,Dutch,"Proficient in Forecasting, Financial Modelling...",Financial Analyst,"As a Financial Analyst, you will be responsibl...",0
3,Keith Hughes,43,Male,Negroid/Black,Caribbean,"Proficient in Budgeting, Supply Chain Optimiza...",Supply Chain Manager,A Supply Chain Manager oversees the entire sup...,1
4,James Davis,49,Male,White/Caucasian,English,"Proficient in Logistics, Negotiation, Procurem...",Supply Chain Manager,A Supply Chain Manager oversees the entire sup...,1


In [23]:
df.tail()

,Job Applicant Name,Age,Gender,Race,Ethnicity,Resume,Job Roles,Job Description,Best Match
9995,Jada Williams,30,Female,Negroid/Black,Ghanaian,"Proficient in Biology, Regulatory Compliance, ...",Biomedical Engineer,A Biomedical Engineer designs and develops med...,0
9996,Jaden Carter,52,Male,Negroid/Black,Nigerian,"Proficient in Communication, Teamwork, Lesson ...",Teacher,A Teacher shapes the future of students by del...,0
9997,Mia Foster,25,Female,White/Caucasian,German,"Proficient in Medical Terminology, Critical Th...",Physician,"Diagnose and treat illnesses, prescribe medica...",0
9998,Stella Green,51,Female,White/Caucasian,Irish,"Proficient in Exercise Programming, Motivation...",Fitness Coach,A Fitness Coach is responsible for helping cl...,1
9999,Ryo Nishida,46,Male,Mongoloid/Asian,Thai,"Proficient in Content Strategy, Copywriting, C...",Content Writer,"As a Content Writer, you will create written m...",0


In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Job Applicant Name  10000 non-null  object
 1   Age                 10000 non-null  int64 
 2   Gender              10000 non-null  object
 3   Race                10000 non-null  object
 4   Ethnicity           10000 non-null  object
 5   Resume              10000 non-null  object
 6   Job Roles           10000 non-null  object
 7   Job Description     10000 non-null  object
 8   Best Match          10000 non-null  int64 
dtypes: int64(2), object(7)
memory usage: 703.2+ KB


In [25]:
df.describe()

,Age,Best Match
count,10000.000000,10000.0000
mean,40.045200,0.4850
std,8.950909,0.4998
min,25.000000,0.0000
25%,32.000000,0.0000
50%,40.000000,0.0000
75%,48.000000,1.0000
max,55.000000,1.0000


In [26]:
df.columns

Index(['Job Applicant Name', 'Age', 'Gender', 'Race', 'Ethnicity', 'Resume',
       'Job Roles', 'Job Description', 'Best Match'],
      dtype='object')

In [27]:
df = df.drop(columns=["Job Applicant Name", "Age", "Gender", "Race", "Ethnicity"])
df["combined_text"] = df["Resume"] + " " + df["Job Description"]

y = df["Best Match"]
X = df["combined_text"]


In [28]:
X

0       Proficient in Injury Prevention, Motivation, N...
1       Proficient in Healthcare, Pharmacology, Medica...
2       Proficient in Forecasting, Financial Modelling...
3       Proficient in Budgeting, Supply Chain Optimiza...
4       Proficient in Logistics, Negotiation, Procurem...
                              ...                        
9995    Proficient in Biology, Regulatory Compliance, ...
9996    Proficient in Communication, Teamwork, Lesson ...
9997    Proficient in Medical Terminology, Critical Th...
9998    Proficient in Exercise Programming, Motivation...
9999    Proficient in Content Strategy, Copywriting, C...
Name: combined_text, Length: 10000, dtype: object

In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [30]:
model = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", max_features=5000)),
    ("clf", LogisticRegression())
])


In [31]:
model.fit(X_train, y_train)


,steps,"[('tfidf', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


In [34]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Accuracy: 0.5245

Confusion Matrix:
 [[537 486]
 [465 512]]

Classification Report:
               precision    recall  f1-score   support

           0       0.54      0.52      0.53      1023
           1       0.51      0.52      0.52       977

    accuracy                           0.52      2000
   macro avg       0.52      0.52      0.52      2000
weighted avg       0.52      0.52      0.52      2000



In [37]:
def check_resume_match(resume_text, job_description):
    combined = resume_text + " " + job_description
    prediction = model.predict([combined])[0]
    
    if prediction == 1:
        print(" Good Match for the Job")
    else:
        print(" Not a Strong Match")


In [38]:
resume_sample = "'Proficient in Budgeting, Supply Chain Optimization, Risk Management, Logistics, Project Management, with senior-level experience in the field. Holds a Masters degree. Holds certifications such as SCPro Certification. Skilled in delivering results and adapting to dynamic environments.'"
job_desc_sample = "'Proficient in Budgeting, Supply Chain Optimization, Risk Management, Logistics, Project Management, with senior-level experience in the field. Holds a Masters degree"

check_resume_match(resume_sample, job_desc_sample)

 Good Match for the Job
